# 🛡️ PhaseGuard - Layer 1 Model Training & Diagnostic Notebook
This interactive notebook allows you to compile the audio datasets, train the **PhaseGuard Layer 1 (AI Voice Authenticity)** model recursively, and visualize the model's performance using metrics such as loss curves, accuracy curves, and confusion matrices.

---

### ⚙️ Step 1: Install Visualization Libraries
Run the cell below to install the necessary plotting and metrics libraries in your active python environment.

In [ ]:
# Install matplotlib, seaborn, and scikit-learn quietly if not already present
!pip install -q matplotlib seaborn scikit-learn
print("Visualization libraries checked and installed successfully!")

### ⚠️ Step 2: Setup Platform-Specific Mocks
Run the cell below to mock heavy libraries (`k2`, `flair`, etc.) that are checked during imports on Windows to prevent runtime import crashes.

In [ ]:
import sys
from unittest.mock import MagicMock
sys.modules['k2'] = MagicMock()
sys.modules['flair'] = MagicMock()
sys.modules['flair.data'] = MagicMock()
sys.modules['flair.embeddings'] = MagicMock()
sys.modules['flair.models'] = MagicMock()
sys.modules['spacy'] = MagicMock()
sys.modules['spacy.tokens'] = MagicMock()

print("Platform mocks applied successfully!")

### Step 3: Import Core ML & Visualization Libraries

In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix, classification_report
from scipy.ndimage import zoom

# Import preprocessing scripts
from preprocess import load_and_standardize, audio_to_mel_spectrogram, extract_5_signals
from train_layer1 import PhaseGuardL1, AudioDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"All packages imported. Device selected: {device}")

### Step 4: Load Preprocessed Dataset Splits
We load the speaker-aware train/test splits generated during preprocessing from `data/processed/`.

In [ ]:
data_path = "data/processed"
train_spec_file = os.path.join(data_path, "train_specs.npy")
train_label_file = os.path.join(data_path, "train_labels.npy")
test_spec_file = os.path.join(data_path, "test_specs.npy")
test_label_file = os.path.join(data_path, "test_labels.npy")

if not (os.path.exists(train_spec_file) and os.path.exists(test_spec_file)):
    raise FileNotFoundError("Dataset splits not found in data/processed! Please run process_phase1.py first to compile splits.")

train_specs = np.load(train_spec_file)
train_labels = np.load(train_label_file)
test_specs = np.load(test_spec_file)
test_labels = np.load(test_label_file)

print("Loaded speaker-aware dataset splits successfully!")
print(f"  Training set: {len(train_labels)} clips (Real: {np.sum(train_labels == 0)}, Fake: {np.sum(train_labels == 1)})")
print(f"  Testing set:  {len(test_labels)} clips (Real: {np.sum(test_labels == 0)}, Fake: {np.sum(test_labels == 1)})")

### Step 5: Define & Run the CNN Model Training Loop
Run this cell to start the **MobileNetV3 CNN** training loop. This will print the progress of each epoch in real time and collect metric history so we can plot graphs.

In [ ]:
# Load datasets
train_dataset = AudioDataset(train_specs, train_labels)
test_dataset = AudioDataset(test_specs, test_labels)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# Instantiate model and training settings
model = PhaseGuardL1().to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

epochs = 20
history = {
    "epoch": [],
    "train_loss": [],
    "train_acc": [],
    "test_loss": [],
    "test_acc": []
}

print("Starting Layer 1 CNN Model Training Loop...")

for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    
    for batch_specs, batch_labels in train_loader:
        batch_specs = batch_specs.to(device)
        batch_labels = batch_labels.to(device)
        
        optimizer.zero_grad()
        predictions = model(batch_specs)
        loss = criterion(predictions, batch_labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * len(batch_labels)
        predicted_classes = (predictions > 0.5).float()
        correct += (predicted_classes == batch_labels).sum().item()
        total += len(batch_labels)
        
    epoch_train_loss = total_loss / total
    epoch_train_acc = (correct / total) * 100
    
    # Evaluate on validation/test splits
    model.eval()
    test_correct = 0
    test_total = 0
    test_loss_sum = 0.0
    
    with torch.no_grad():
        for batch_specs, batch_labels in test_loader:
            batch_specs = batch_specs.to(device)
            batch_labels = batch_labels.to(device)
            
            predictions = model(batch_specs)
            loss = criterion(predictions, batch_labels)
            test_loss_sum += loss.item() * len(batch_labels)
            
            predicted_classes = (predictions > 0.5).float()
            test_correct += (predicted_classes == batch_labels).sum().item()
            test_total += len(batch_labels)
            
    epoch_test_loss = test_loss_sum / test_total
    epoch_test_acc = (test_correct / test_total) * 100
    
    # Record history
    history["epoch"].append(epoch + 1)
    history["train_loss"].append(epoch_train_loss)
    history["train_acc"].append(epoch_train_acc)
    history["test_loss"].append(epoch_test_loss)
    history["test_acc"].append(epoch_test_acc)
    
    print(f"Epoch {epoch+1:02d}/{epochs:02d} | Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.1f}% | Test Loss: {epoch_test_loss:.4f} | Test Acc: {epoch_test_acc:.1f}%")

# Save trained weights
os.makedirs("models", exist_ok=True)
torch.save(model.state_dict(), "models/layer1_mobilenet.pth")
print("\nLayer 1 training complete! Weights saved successfully to models/layer1_mobilenet.pth")

### Step 6: Plot Loss & Accuracy Curves
Run this cell to render training progress charts showing the decreasing loss and increasing accuracy over time.

In [ ]:
plt.figure(figsize=(14, 5))

# Plot Loss curve
plt.subplot(1, 2, 1)
plt.plot(history["epoch"], history["train_loss"], 'o-', label='Train Loss', color='#e74c3c')
plt.plot(history["epoch"], history["test_loss"], 's--', label='Test/Val Loss', color='#2980b9')
plt.title('Training & Testing Loss Curve', fontsize=12, fontweight='bold')
plt.xlabel('Epochs', fontsize=10)
plt.ylabel('Loss Value', fontsize=10)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(frameon=True, facecolor='white')

# Plot Accuracy curve
plt.subplot(1, 2, 2)
plt.plot(history["epoch"], history["train_acc"], 'o-', label='Train Accuracy', color='#2ecc71')
plt.plot(history["epoch"], history["test_acc"], 's--', label='Test/Val Accuracy', color='#9b59b6')
plt.title('Training & Testing Accuracy Curve', fontsize=12, fontweight='bold')
plt.xlabel('Epochs', fontsize=10)
plt.ylabel('Accuracy (%)', fontsize=10)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(frameon=True, facecolor='white')

plt.tight_layout()
plt.show()

### Step 7: Confusion Matrix & Classification Report
Generate a beautiful confusion matrix heatmap and print precision, recall, and f1-score measurements for both Real (0) and Fake (1) voices.

In [ ]:
# Set model to evaluation mode
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch_specs, batch_labels in test_loader:
        batch_specs = batch_specs.to(device)
        predictions = model(batch_specs)
        predicted_classes = (predictions > 0.5).float().cpu().numpy()
        
        all_preds.extend(predicted_classes)
        all_labels.extend(batch_labels.cpu().numpy())

all_preds = np.array(all_preds).squeeze()
all_labels = np.array(all_labels).squeeze()

# Calculate confusion matrix
cm = confusion_matrix(all_labels, all_preds)

# Plot using Seaborn
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Real (0)', 'Fake/AI (1)'], 
            yticklabels=['Real (0)', 'Fake/AI (1)'],
            cbar=False, annot_kws={"size": 14, "weight": "bold"})
plt.title('Test Set Confusion Matrix', fontsize=13, fontweight='bold', pad=15)
plt.ylabel('Actual Category', fontsize=11)
plt.xlabel('Predicted Category', fontsize=11)
plt.show()

# Print classification report
print("\n" + "="*50)
print("           CNN CLASSIFICATION REPORT")
print("="*50)
print(classification_report(all_labels, all_preds, target_names=['Real (Human)', 'Fake (Clone)']))
print("="*50)

### Step 8: Manually Test Individual Audio Files (Inference Widget)
Change the audio path variable in this cell and run it to classify any specific audio file as real or fake!

In [ ]:
# ENTER THE AUDIO FILE PATH YOU WANT TO TEST HERE
test_file_path = "data/real_voices/abhinav/abhinav_001.wav"

if not os.path.exists(test_file_path):
    print(f"File not found: {test_file_path}. Please check the path and try again.")
else:
    print(f"Analyzing clip: {test_file_path}...")
    
    # Standardize sample to 2s clip
    audio = load_and_standardize(test_file_path)
    mel = audio_to_mel_spectrogram(audio)
    
    # Match dimensions
    if mel.shape[1] != 128:
        mel_resized = zoom(mel, (1, 128 / mel.shape[1]))
    else:
        mel_resized = mel
        
    mel_norm = (mel_resized - mel_resized.min()) / (mel_resized.max() - mel_resized.min() + 1e-10)
    tensor = torch.FloatTensor(mel_norm).unsqueeze(0).unsqueeze(0).to(device)
    
    model.eval()
    with torch.no_grad():
        prediction = model(tensor).item()
        
    # Extract acoustic physical signals
    feats = extract_5_signals(audio)
    
    print("\n" + "="*60)
    print("                  PHASEGUARD INFERENCE DIAGNOSTIC")
    print("="*60)
    print(f"Audio File        : {os.path.basename(test_file_path)}")
    print(f"AI Confidence     : {prediction * 100:.2f}%")
    
    if prediction > 0.5:
        print(f"DECISION          : [ALERT] FAKE / AI VOICE CLONE (Probability: {prediction:.4f})")
    else:
        print(f"DECISION          : [OK] REAL HUMAN VOICE (Probability: {prediction:.4f})")
        
    print("-"*60)
    print("               Acoustic Signal Extraction Details")
    print("-"*60)
    print(f"Phase Jump Rate     : {feats['phase_jump_rate']:.4f}")
    print(f"Pitch Jitter        : {feats['jitter']:.6f}")
    print(f"Spectral Flatness   : {feats['spectral_flatness']:.4f}")
    print(f"Noise Floor (RMS)   : {feats['noise_floor']:.6f}")
    print(f"MFCC Delta Variance : {feats['mfcc_delta_var']:.4f}")
    print("="*60)

---

## 📖 Instructions: How to Run This Notebook

Follow these steps to demonstrate the model training live to others:

1. **Activate the Conda Environment**:
   Open **Anaconda Prompt** or terminal, and run:
   ```powershell
   conda activate phaseguard
   ```
2. **Open Jupyter Notebook**:
   Navigate to this workspace folder (`E:\UCO-HACKATHON`) and type:
   ```powershell
   jupyter notebook
   ```
3. **Select the Kernel**:
   Once the notebook opens in your web browser, verify the top-right kernel says **`Python (PhaseGuard)`** or your primary conda Python kernel.
4. **Run Cells Sequentially**:
   * Run **Step 1** to make sure plotting tools are installed.
   * Run **Step 2 & 3** to set up mocks and imports.
   * Run **Step 4** to load the dataset splits.
   * Run **Step 5** to watch the MobileNetV3 CNN train in real time across the 20 epochs.
   * Run **Step 6 & 7** to show the learning curve graphs, the final Confusion Matrix heatmap, and classification score details.
   * Run **Step 8** to run inference on any test `.wav` audio files you select.